# README

- **Author**: RUEI-HSIANG, CHANG（B12303030；AI協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?

整合續查社群入口、逐頁學經歷原文、林德建官方生日及縣市長首長名錄比對；另將第三方履歷主張展開為待查工作單。候選人主鍵不變，不刪人、不由缺漏填零。社群只讀候選人專屬區塊；不把分享網址、媒體與政黨共用頁尾當作個人帳號。學經歷保留來源原文、不認證學位；歷史首次參選僅在有過往參選證據時記0，不從沒找到紀錄推定首次=1。性別不從職業／照片／姓名推測。

## What does this file take?

每個具名輸入及固定SHA-256見下方PINS。前輪current快照為輸入；raw manifest記錄每份HTTP來源bytes與hash，讀取前逐檔核對。原始來源說明見 `data/raw/career_round6_20261019_documentation.ipynb`。

## What does this file output?

- 五個current檔：1583列master、31660列field、1502列council、逐來源帳號證據及4749列社群工作單。
- `candidate_profiles_round6_20261019.csv`：18人逐頁背景原文。
- `candidate_background_evidence_round6_20261019.csv`：候選人×欄位×來源觀察。
- `mayor_head_comparison_round6_20261019.csv`：81人與22縣市名錄首長姓名比對；不是完整代理／停職法律狀態審核。
- `candidates_2026_enriched_round6_20261019.csv`：81人，原欄位不刪除；新增名錄比對與來源，僅8個正向首長身分有近期官方來源支持。
- `council_career_source_worklist_round6_20261019.csv`：第三方主張逐筆來源查找狀態；不是已核實個人經歷。

輸出角色：review_inventory／source_observation／discovery_worklist。整理後說明見 `data/processed/career_round6_20261019_documentation.ipynb`。精確輸出路徑、筆數、SHA-256於最後一格以讀回結果列示並保存在verification.json；不供直接推論或訓練模型。


### 本輪續補

2026-10-19重新核對陳重嘉官方議會人物頁，明載完整生日1981-01-19與歷屆議員經歷，接回81人縣市長表及current欄位狀態；完整生日由57增至58人，仍缺23人。讀取先前保存的官方HTML；FOLLOWUP_PINS固定原檔及擷取metadata hash，不宣稱HTTP重新下載成功。

温世政1973-05-04續補：2021-07-17專業學會名冊第1頁第8列已視讀，以中央社人物生日欄與NOWnews當屆診所／參選身分交叉支持；非政府來源。兩筆直接HTTP403保留，另存web工具短摘錄，不冒充原始HTTP成功。縣市長完整生日有不同來源層級支持者59人、仍缺22人。

第七輪接續註記：本輪三份current背景／逐欄輸出已封存至pre_identity_round7快照；本文件仍讀寫其第六輪版本，最新整合由council_identity_round7產生。

In [1]:
from pathlib import Path
import pandas as pd,json,re,hashlib,unicodedata
from bs4 import BeautifulSoup
from urllib.parse import urlparse,urlsplit,urlunsplit,parse_qs,urlencode
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def read(p):return pd.read_csv(ROOT/p,dtype=str,keep_default_na=False)
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',str(s))).replace('台','臺').replace('温','溫')
PINS={'data/processed/snapshots/pre_career_round6_20261019/candidate_master_current.csv': '3466e59efffd916b6226f7ce99fa6df2a93c4e0a879697efef530e764492073d', 'data/processed/snapshots/pre_career_round6_20261019/candidate_fields_current.csv': 'e945fc9003be3c9a4cffbaa6593bce433bf117ae6d5f85fea8efcd3d681b7666', 'data/processed/snapshots/pre_career_round6_20261019/council_background_20261019.csv': '7557d14728f92fa25879d906a5ba65b00598bda38c9328c829a483ed4e69d2fe', 'data/processed/snapshots/pre_career_round6_20261019/candidate_social_account_evidence_current.csv': '1d201d2feb23f6813126ccd52959c5d5b43ad8a70dbe5b447ca820f833f2e188', 'data/processed/snapshots/pre_career_round6_20261019/meta_candidate_account_worklist_current.csv': 'e8e890fd04a1dc4970ca31685831be1a77f1e645d959ba0ca49c9d5676869f5b', 'data/processed/candidates_2026_enriched_round4_20261019.csv': '37fa15d5298ba12857f55fdd87d3d3903a1dc68d7eb975995b6d228f92be4194', 'data/raw/social_followup_round6_20261019/manifest.json': 'f5471133039402a9e21143f23d30b70b645ddf3406805e19f78764f5b2063746', 'data/raw/career_expansion_20261019/manifest.json': '9db18b5ba806b96a064ad2e4c317aa3fd6f57a7ca66e7bef64c51f47411fb09e', 'data/raw/career_expansion_20261019/retrieval_plan.json': '88f06be25b2c237e1644ada60c7f654f83ca3f4ef4a414e070c7551aeb01c5f4', 'data/raw/mayor_incumbency_20261019/manifest.json': 'bf8428c29a8404f6229ac6975b709f05193af37d8025cffbfc78bd712af808e0'}
for p,h in PINS.items():assert (ROOT/p).is_file() and sha(ROOT/p)==h,p
S='data/processed/snapshots/pre_career_round6_20261019'
master=read(S+'/candidate_master_current.csv');fields=read(S+'/candidate_fields_current.csv');council=read(S+'/council_background_20261019.csv');accounts=read(S+'/candidate_social_account_evidence_current.csv');work=read(S+'/meta_candidate_account_worklist_current.csv');mayors=read('data/processed/candidates_2026_enriched_round4_20261019.csv')
assert len(master)==1583 and master.candidate_key.is_unique and len(fields)==31660
assert len(council)==1502 and len(mayors)==81 and len(work)==4749
captures=[]
for p in ['data/raw/social_followup_round6_20261019/manifest.json','data/raw/career_expansion_20261019/manifest.json','data/raw/mayor_incumbency_20261019/manifest.json']:
 captures+=json.loads((ROOT/p).read_text())
for a in captures:
 if a.get('path'):assert (ROOT/a['path']).is_file() and sha(ROOT/a['path'])==a['sha256'],a['url']
byurl={a['url']:a for a in captures}
def source(u):
 a=byurl[u];assert str(a['status']) in ['200','existing_preserved_capture'],u
 return BeautifulSoup((ROOT/a['path']).read_bytes().decode('utf-8',errors='replace'),'html.parser')
def text(u):return re.sub(r'\s+',' ',source(u).get_text(' ',strip=True).replace('\x00','')).strip()
def person(n):
 rows=master[master.candidate_name.eq(n)];assert len(rows)==1,n
 return rows.iloc[0]
evidence=[]
def obs(n,field,value,u,note,year='2026',key=None):
 p=person(n) if key is None else master[master.candidate_key.eq(key)].iloc[0]
 assert p.candidate_name==n
 evidence.append({'candidate_key':p.candidate_key,'candidate_name':n,'field':field,'value':value,'source_url':u,'source_file':byurl[u]['path'],'source_year':year,'evidence_status':note,'checked_date':'2026-10-19'})
def setfield(key,f,value,status,src,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(f);assert mask.sum()==1
 fields.loc[mask,['existing_value','status','evidence','note']]=[value,status,src,note]
 master.loc[master.candidate_key.eq(key),f+'_status']=status
print('Bound input files:',len(PINS),'captured source attempts:',len(captures))

HISTORY_PINS={'data/raw/cec_all_offices_manifest.json': '726d464020ff8f850ce39685bb70926c8529712f75c1833ac5b5c6d795354ed7', 'data/raw/cec_all_offices/068_elcand.csv': '47ee3b569c7bd6d2d0e989d087457a10dc4a7c2657df51c9c1c97a6436f96720', 'data/raw/cec_all_offices/080_elcand.csv': '19cfbfb2e639ff2e5310856ec64e7933b6d4ff483dc09bd73092075ba12db6e7', 'data/raw/cec_all_offices/095_elcand.csv': '2106bc9400f7d64784595799d2404496186cffaa56d82b14554d172d781f1b60', 'data/raw/cec_all_offices/116_elcand.csv': 'cf9a675fd5d26c6a6f2961f3c0654699c0fe7905c81a2e68d26d9f3cd852ee3d'}
for p,h in HISTORY_PINS.items():assert (ROOT/p).is_file() and sha(ROOT/p)==h,p

FOLLOWUP_PINS={'data/raw/sex_verification/chen_zhongjia_retry.html': 'c10bbe149bb92452cb1793ef3fe19d3483cb14254765768fbb1fee314c732c37', 'data/raw/sex_verification/capture_manifest_2.json': 'f0149f02a44191d0c2d0b4540ea2501a9d21070b8bd40c39c46d7fe7bfd9cf55'}
for p,h in FOLLOWUP_PINS.items():assert (ROOT/p).is_file() and sha(ROOT/p)==h,p
byurl['https://www.chcc.gov.tw/member/details.aspx?Parser=99,6,40,,,,142']={'url': 'https://www.chcc.gov.tw/member/details.aspx?Parser=99,6,40,,,,142', 'status': 'existing_preserved_capture', 'path': 'data/raw/sex_verification/chen_zhongjia_retry.html', 'sha256': 'c10bbe149bb92452cb1793ef3fe19d3483cb14254765768fbb1fee314c732c37', 'capture_metadata': 'data/raw/sex_verification/capture_manifest_2.json', 'retrieval_note': 'Previously saved official HTML; original metadata has no HTTP-status field. Current webpage independently rechecked with web tool 2026-10-19; requests retry failed certificate validation, not recorded as a new successful fetch.'}

WEN_PINS={'data/raw/sex_verification/wen_shizheng.pdf': '18316813b98dcf6a59c1afe8f194ad9377515e94ce5d54f26d9a3eae2cf1025c', 'data/raw/mayor_birth_followup_20261019/manifest.json': 'c6bb81a76aa0641236ab21dd10f6085731756a1d75dc1808a7c61c4ab543634c', 'data/raw/mayor_birth_followup_20261019/wen_cna.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/mayor_birth_followup_20261019/wen_nownews.html': 'e25c358cf17143a8055b5acf2cf676669b4aac553d128df421d502f01440d8a4', 'data/raw/mayor_birth_followup_20261019/web_tool_excerpts.json': '82a726fcc4a7ffe05c812300cb87d488b7683bfc412188313dcde404440dedf2'}
for p,h in WEN_PINS.items():assert (ROOT/p).is_file() and sha(ROOT/p)==h,p
byurl['https://x.miniwork.cc/manager_admin/new_file_download.php?Pact=FileDownLoad&Pval=426898']={'url': 'https://x.miniwork.cc/manager_admin/new_file_download.php?Pact=FileDownLoad&Pval=426898', 'status': 'existing_preserved_capture', 'path': 'data/raw/sex_verification/wen_shizheng.pdf', 'sha256': '18316813b98dcf6a59c1afe8f194ad9377515e94ce5d54f26d9a3eae2cf1025c', 'source_year': '2021', 'source_tier': 'professional_association_candidate_roster_not_government', 'document_date': '2021-07-17', 'locator': 'page 1 row 8; visually checked 2026-10-19'}


Bound input files: 10 captured source attempts: 260


In [2]:

# Candidate-specific source sections; never extract organization footer links.
profiles=[];new_accounts=[]
def canonical(h):
 z=urlsplit(h);query=''
 if z.hostname and 'facebook.com' in z.hostname and z.path.rstrip('/')=='/profile.php':query=urlencode({'id':parse_qs(z.query)['id'][0]})
 return urlunsplit((z.scheme,z.netloc,z.path.rstrip('/'),query,''))
def social(n,u,anchors,locator):
 p=person(n)
 for a in anchors:
  h=a.get('href','');host=urlparse(h).hostname or ''
  plat=next((plat for domain,plat in [('facebook.com','Facebook'),('instagram.com','Instagram'),('threads.com','Threads'),('threads.net','Threads')] if host==domain or host.endswith('.'+domain)),'')
  if not plat or any(x in urlparse(h).path for x in ['/share/','/sharer','/groups/','/posts/','/reel','/dialog/']):continue
  new_accounts.append({'candidate_key':p.candidate_key,'candidate_name':n,'county_city':p.county_city,'office_type':p.office_type,'platform':plat,'account_url':canonical(h),'source_url':u,'source_file':byurl[u]['path'],'source_locator':locator,'evidence_status':'candidate_specific_source_link_target_content_pending','target_content_status':'not_independently_read','checked_date':'2026-10-19','source_year':'2026','source_identity_status':'name_and_current_county_district_or_service_area_context'})
def profile(n,u,edu,exp,tier,note):
 p=person(n);raw=text(u);assert norm(n) in norm(raw)
 for field,value in [('education',edu),('experience',exp)]:
  for part in value.split(' | '):
   if part:assert norm(part) in norm(raw),(n,field,part)
  if value:obs(n,field,value,u,tier+'；'+note)
 profiles.append({'candidate_key':p.candidate_key,'candidate_name':n,'county_city':p.county_city,'electoral_district':p.electoral_district,'source_url':u,'source_file':byurl[u]['path'],'source_tier':tier,'education_raw':edu,'experience_raw':exp,'identity_review':note,'checked_date':'2026-10-19'})
for i in range(1,12):
 u=f'https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-{i:03d}';s=source(u);name=s.h1.get_text(' ',strip=True);p=person(name);raw=text(u)
 assert norm(p.county_city) in norm(raw[:250]),name
 a=raw.index('主要經歷');b=raw.index('核心政見',a);exp=raw[a+len('主要經歷'):b].strip();assert 0<len(exp)<1400
 edu='海軍陸戰隊專修班45期畢 | 醒吾科技大學 | 省立仁愛高農' if name=='林啟明' else ''
 profile(name,u,edu,exp,'party_candidate_page_self_reported','姓名、當屆縣市及選區文字與名冊一致；曾任／現任按來源原文保留，不單憑此認定任職法律狀態')
 social(name,u,s.select('.hero-social-box a[href]'),'candidate hero-social-box; shared footer excluded')
review=[
 ('張柏詠','https://ryung.oen.tw/','','過去在調查局工作的8年','campaign_self_reported','姓名、新北市議員、新店服務地區一致'),
 ('何元楷','https://teamho666.oen.tw/','','曾擔任蔣萬安市長的幕僚、張善政市長的發言人，現任為立委羅智強辦公室主任','campaign_self_reported','姓名與汐止、萬里、金山參選地區一致'),
 ('白丞堯','https://votepai.oen.tw/','','白丞堯律師','campaign_self_reported','姓名與桃園平鎮服務處一致；律師為職業自述，不推論學位或證照驗證'),
 ('鄒約翰','https://john.oen.tw/','','青年時，我穿上軍裝保家衛國。 | 1999年，成立「高雄宋楚瑜正義之友會」，呼群保義。','campaign_self_reported','姓名與高雄第8選區一致；未由軍旅經歷推論性別'),
 ('謝秉舟','https://www.arkbiomedical.org/councilor','','牙醫出身，長期關注社區口腔健康與高齡照護。','campaign_self_reported','姓名與臺南中西區、南區服務處一致；未將職業自述當作學位認證'),
 ('潘炩禕','https://panlingyi.tw/','奧地利留學回國','屏東縣文化協會理事長','campaign_self_reported','姓名及屏東第三選區服務區一致；留學不當作畢業或特定學位'),
 ('林德建','https://www.dongyin.gov.tw/Chhtml/content/2499','東引國中小、東山高中','東引鄉代會第8屆主席 | 連江縣議會機要秘書 | 馬祖連江航業董事長','government_person_profile','鄉長人物頁及2026議會登記報導交叉支持；生日與性別分開，不由職務推論性別')]
for row in review:
 profile(*row)
 if row[0] in ['張柏詠','何元楷','白丞堯','鄒約翰']:social(row[0],row[1],source(row[1]).select('a[href]'),'candidate OEN page social links; sharing URLs excluded')
 if row[0]=='謝秉舟':social(row[0],row[1],source(row[1]).select('[aria-label="社群媒體"] a[href]'),'candidate social block href and aria-label; icon shape ignored')
 if row[0]=='林德建':
  assert 'https://www.facebook.com/mjstard' in text(row[1])
  social(row[0],row[1],[{'href':'https://www.facebook.com/mjstard'}],'official township head profile explicitly printed URL as plain text, not an HTML anchor')
profiles=pd.DataFrame(profiles);assert len(profiles)==18 and profiles.candidate_key.is_unique
# Extra explicit birth evidence; keep separate from the 2022 supported-history series.
u='https://www.dongyin.gov.tw/Chhtml/content/2499';raw=text(u);assert '民國60年5月4日' in raw
identity_url='https://www.mtcc.gov.tw/ch/Detail/7168?mcid=113316&webaid='
assert '東引鄉長林德建' in text(identity_url) and '完成登記' in text(identity_url)
obs('林德建','birth_date','1971-05-04',u,'官方人物頁生日；民國轉西元；2026議會登記報導交叉連結')
obs('林德建','identity_link','東引鄉長林德建',identity_url,'議會2026-09-05登記報導，支持跨表身分')
for col in ['birth_date_additional_official','birth_date_additional_source_url','birth_date_additional_evidence_status','round6_profile_urls']:
 council[col]=''
key=person('林德建').candidate_key;council.loc[council.candidate_key.eq(key),['birth_date_additional_official','birth_date_additional_source_url','birth_date_additional_evidence_status']]=['1971-05-04',u,'official_profile_plus_current_council_registration_identity_support']
setfield(key,'birth_date','1971-05-04','官方人物頁生日與當屆參選身分有交叉支持',u+' | '+identity_url,'民國60年5月4日轉西元；不覆寫2022歷史序列；性別仍未知')
new_profile_keys=[]
for a in profiles.to_dict('records'):
 key=a['candidate_key'];mask=council.candidate_key.eq(key);row=council[mask].iloc[0]
 council.loc[mask,'round6_profile_urls']=a['source_url']
 if not any(row[c] for c in ['moi_profile_url','campaign_profile_url','additional_profile_url']):
  new_profile_keys.append(key);council.loc[mask,['additional_profile_url','additional_profile_status']]=[a['source_url'],a['source_tier']+'; '+a['identity_review']]
  setfield(key,'profile',a['source_url'],'人物頁已有，欄位與身分核對狀態分列',a['source_url'],a['identity_review'])
 for field,col in [('education','education_raw'),('experience','experience_raw')]:
  if a[col] and not row[col]:
   council.loc[mask,col]=a[col]
   if not row['background_source_url']:council.loc[mask,'background_source_url']=a['source_url']
   if field=='education':setfield(key,field,a[col],'已有來源原文，未認證畢業或學位',a['source_url'],a['source_tier']+'；'+a['identity_review'])
 # Website is a source URL, not a claim that a social account was directly read.
 if a['source_tier']=='campaign_self_reported':
  old=fields[(fields.candidate_key.eq(key))&fields.field.eq('website')].iloc[0]
  if not old.existing_value:setfield(key,'website',a['source_url'],'候選人競選／服務網站來源已讀',a['source_url'],a['identity_review'])
# Only supported negative first-run flags. No positive flag from search absence.
first_evidence={
 '陳泓維':('2020 桃園市第一選區立委參選人','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-003'),
 '陳傑麟':('宜蘭縣議會第18屆、第19屆縣議員','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-005'),
 '林啟明':('2018 新北市烏來區區民代表候選人','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-010'),
 '吳建智咖啡哥':('2024 平地原住民立法委員候選人','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-011'),
 '林德建':('東引鄉代會第8屆主席','https://www.dongyin.gov.tw/Chhtml/content/2499')}
new_nonfirst_any=[];new_nonfirst_council=[]
for name,(claim,u) in first_evidence.items():
 assert norm(claim) in norm(text(u));key=person(name).candidate_key;mask=council.candidate_key.eq(key)
 if council.loc[mask,'first_time_any_office_supported'].iloc[0]=='':new_nonfirst_any.append(name)
 council.loc[mask,'first_time_any_office_supported']='0';obs(name,'first_time_any_office','0',u,'有過往民選公職參選／任職原文：'+claim)
 setfield(key,'first_time_any_office','0','有來源支持非終生首次參選',u,claim+'；政黨頁為自述證據，未由缺漏反推首次')
 if name=='陳傑麟':
  if council.loc[mask,'first_time_this_office_supported'].iloc[0]=='':new_nonfirst_council.append(name)
  council.loc[mask,'first_time_this_office_supported']='0';obs(name,'first_time_this_office','0',u,'曾任地方議員，支持非首次參選地方議員')
  setfield(key,'first_time_this_office','0','有來源支持非首次參選議員',u,claim)
# Source evidence and per-person/platform worklist.
new_accounts=pd.DataFrame(new_accounts).drop_duplicates(['candidate_key','platform','account_url','source_url'])
assert len(new_accounts)==29 and new_accounts.candidate_key.nunique()==16
accounts=pd.concat([accounts,new_accounts],ignore_index=True).drop_duplicates(['candidate_key','platform','account_url','source_url'])
for (key,plat),d in new_accounts.groupby(['candidate_key','platform']):
 allrows=accounts[accounts.candidate_key.eq(key)&accounts.platform.eq(plat)];mask=work.candidate_key.eq(key)&work.platform.eq(plat);assert mask.sum()==1
 urls=' | '.join(dict.fromkeys(allrows.account_url));src=' | '.join(dict.fromkeys(allrows.source_url))
 work.loc[mask,['official_urls','discovered_on','account_status']]=[urls,src,'candidate_specific_source_link_target_content_pending']
 setfield(key,plat,urls,'來源頁列出候選人帳號，目標內容待核對',src,'未取得完整貼文；HTTP存取限制不當作零貼文')
assert accounts[['candidate_key','platform']].drop_duplicates().shape[0]==201 and accounts.candidate_key.nunique()==87
assert work[['window_count','followers_start','followers_end']].eq('').all().all()
print('Profile pages',len(profiles),'new profile people',len(new_profile_keys),'social newly sourced',len(new_accounts),'first-any new',new_nonfirst_any)


Profile pages 18 new profile people 16 social newly sourced 29 first-any new ['陳泓維', '陳傑麟', '林啟明', '吳建智咖啡哥', '林德建']


In [3]:

# Historical official declarations linked via explicit past-election/career context.
import csv,datetime
archive=json.loads((ROOT/'data/raw/cec_all_offices_manifest.json').read_text());members={a['path']:a for a in archive['members']}
for col in ['birth_date_additional_source_year','additional_historical_education_cec']:
 council[col]=''
council.loc[council.candidate_name.eq('林德建'),'birth_date_additional_source_year']='2026'
history_links=[
 ('陳泓維','data/raw/cec_all_offices/095_elcand.csv',2020,'2020總統立委/區域立委','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-003',['68','000','01']),
 ('林啟明','data/raw/cec_all_offices/068_elcand.csv',2014,'直轄市區長','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-010',['65','000','01','290']),
 ('吳建智咖啡哥','data/raw/cec_all_offices/116_elcand.csv',2024,'2024總統立委/平地立委','https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-011',['00','000','01'])]
linked=[]
for name,path,year,context,anchor,geog in history_links:
 assert context in members[path]['archive_member'] and sha(ROOT/path)==members[path]['sha256']
 rows=list(csv.reader((ROOT/path).open(encoding='utf-8-sig')))
 hits=[(i+1,[v.strip().lstrip("'") for v in row]) for i,row in enumerate(rows) if len(row)>12 and row[6].strip().lstrip("'")==name]
 assert len(hits)==1,(name,path);line,row=hits[0]
 assert row[:len(geog)]==geog and row[8] in ['1','2'] and re.fullmatch(r'\d{7}',row[9])
 birth=datetime.date(int(row[9][:3])+1911,int(row[9][3:5]),int(row[9][5:])).isoformat();sex={'1':'男','2':'女'}[row[8]]
 if name=='林啟明':
  # The same 1962 birthday appears in the 2018 district-representative file;
  # other same-name candidates in different counties/years are not linked.
  path2='data/raw/cec_all_offices/080_elcand.csv';assert '2018-' in members[path2]['archive_member']
  matches=[[v.strip().lstrip("'") for v in rr] for rr in csv.reader((ROOT/path2).open(encoding='utf-8-sig')) if len(rr)>12 and rr[6].strip().lstrip("'")==name]
  assert len(matches)==1 and matches[0][9]==row[9] and matches[0][:4]==['65','000','01','290']
 key=person(name).candidate_key;mask=council.candidate_key.eq(key)
 assert council.loc[mask,'birth_date_supported_history'].iloc[0]==''
 assert council.loc[mask,'sex_supported_history'].iloc[0]==''
 council.loc[mask,['birth_date_additional_official','birth_date_additional_source_url','birth_date_additional_evidence_status','birth_date_additional_source_year','sex_additional_source_value','sex_additional_source_url','sex_additional_source_year','additional_historical_education_cec']]=[birth,'https://db.cec.gov.tw/',f'{year} official declaration; identity linked by explicit party-page election-year/office/county context',str(year),sex,'https://db.cec.gov.tw/',str(year),row[12]]
 for field,value in [('birth_date',birth),('sex',sex),('education_history',row[12])]:
  evidence.append({'candidate_key':key,'candidate_name':name,'field':field,'value':value,'source_url':'https://db.cec.gov.tw/','source_file':path,'source_year':str(year),'evidence_status':f'official historical declaration line {line}; '+context+'; identity anchor '+anchor,'checked_date':'2026-10-19'})
  if field in ['birth_date','sex']:setfield(key,field,value,f'{year}中選會歷史申報，當屆履歷有跨表身分支持','https://db.cec.gov.tw/ | '+path+':'+str(line)+' | '+anchor,'保留來源年代；不混入2022序列，不視為2026戶籍重新認證')
 linked.append({'candidate_name':name,'source_file':path,'source_row':line,'source_year':year,'birth_date':birth,'sex':sex,'education_raw':row[12]})
print('New historical birthday/sex links:',linked)


New historical birthday/sex links: [{'candidate_name': '陳泓維', 'source_file': 'data/raw/cec_all_offices/095_elcand.csv', 'source_row': 256, 'source_year': 2020, 'birth_date': '1987-05-08', 'sex': '男', 'education_raw': '高中(職)'}, {'candidate_name': '林啟明', 'source_file': 'data/raw/cec_all_offices/068_elcand.csv', 'source_row': 3, 'source_year': 2014, 'birth_date': '1962-05-19', 'sex': '男', 'education_raw': '專科'}, {'candidate_name': '吳建智咖啡哥', 'source_file': 'data/raw/cec_all_offices/116_elcand.csv', 'source_row': 6, 'source_year': 2024, 'birth_date': '1982-05-12', 'sex': '男', 'education_raw': '高中(職)'}]


In [4]:

# Separate the elected-head directory, observed acting head, and candidate matching.
gov_urls={
 '臺北市':'https://www.gov.taipei/News_Content_Leader.aspx?n=5A0C91ADEE6E7C46&s=5C77853F1AB78232&sms=7CAF6BD4D3E48630',
 '新北市':'https://www.ntpc.gov.tw/ch/home.jsp?id=5de1e44dcb4cd218',
 '桃園市':'https://idipc-tiiai.tycg.gov.tw/about?id=0svttlticq8f56dl',
 '臺中市':'https://cdn.www.taichung.gov.tw/8868/9942/9986/9989/1173576/post',
 '臺南市':'https://www.tainan.gov.tw/cp.aspx?n=5034',
 '高雄市':'https://www.kcg.gov.tw/cp.aspx?n=59C096A5BBC7AFB1',
 '新竹縣':'https://www.hsinchu.gov.tw/cp.aspx?n=134',
 '苗栗縣':'https://www.miaoli.gov.tw/cp.aspx?n=265',
 '彰化縣':'https://www.chcg.gov.tw/ch2/intro-magistrate.aspx',
 '南投縣':'https://www.nantou.gov.tw/big5/introduction-magistrate.php',
 '宜蘭縣':'https://www.e-land.gov.tw/cp.aspx?n=2700',
 '嘉義縣':'https://www.president.gov.tw/News/40088',
 '嘉義市':'https://www.chiayi.gov.tw/cp.aspx?n=446',
 '屏東縣':'https://www.pthg.gov.tw/cp.aspx?n=2CD2D022F4AA2FE5',
 '臺東縣':'https://www.taitung.gov.tw/cp.aspx?n=13358',
 '花蓮縣':'https://www.hl.gov.tw/cp.aspx?n=32793',
 '澎湖縣':'https://www.penghu.gov.tw/ch/home.jsp?id=10048',
 '基隆市':'https://www.klcg.gov.tw/tw/klcg1/3176.html',
 '新竹市':'https://www.hccg.gov.tw/',
 '金門縣':'https://www.kmdn.gov.tw/1117/1271/1272/588384/',
 '連江縣':'https://www.matsu.gov.tw/chhtml/chiefpage/371030000A/19'}
heads={}
for a in json.loads((ROOT/'data/raw/mayor_incumbency_20261019/manifest.json').read_text()):
 if 'LocalOfficial_Content' not in a['url'] or str(a['status'])!='200':continue
 s=source(a['url']);agency=s.select_one('.lead-data .figcaption .title').get_text(' ',strip=True);county=agency.replace('政府','');name=s.select_one('.lead-data .figcaption .name span').get_text(' ',strip=True)
 assert county not in heads
 role_name='林茂盛' if county=='宜蘭縣' else name
 u=gov_urls.get(county,'');context='';locator=''
 if u:
  s=source(u);body=text(u);alts=' '.join(x.get('alt','') for x in s.select('img[alt]'));body+=' '+alts
  assert norm(role_name) in norm(body),(county,role_name,u)
  i=body.find(role_name);context=body[max(0,i-80):i+200];locator='HTML text and explicitly named img alt; not inferred from image face'
  if county=='新竹市':assert '115-10-19' in body and '高虹安市長' in body
  if county=='宜蘭縣':assert '代理縣長' in body and '113年12月31日迄今' in body
 heads[county]={'listed_elected_head_name':name,'moi_source_url':a['url'],'moi_source_file':a['path'],'government_role_name_as_captured':role_name if u else '', 'government_role_source_url':u,'government_role_source_file':byurl[u]['path'] if u else '', 'government_role_context':context,'government_role_locator':locator,'source_scope':'official_role_context_as_captured_not_full_legal_status_audit' if u else 'moi_directory_only_current_role_recheck_pending','acting_head_observed':'1' if county=='宜蘭縣' else ''}
assert len(heads)==22 and set(mayors.county_city)==set(heads)
comparisons=[];incumbent_positive=[]
for p in mayors.to_dict('records'):
 h=heads[p['county_city']];match=int(norm(p['candidate_name'])==norm(h['listed_elected_head_name']));registered=int(any(norm(x)==norm(h['listed_elected_head_name']) for x in mayors[mayors.county_city.eq(p['county_city'])].candidate_name))
 comparisons.append({'candidate_key':p['candidate_key'],'candidate_name':p['candidate_name'],'county_city':p['county_city'],**h,'matches_listed_elected_head':str(match),'listed_elected_head_registered_in_roster':str(registered),'comparison_status':'matches_listed_elected_head_and_official_role_context' if match else 'not_the_listed_elected_head; no automatic term_limit_or_vacancy_inference','checked_date':'2026-10-19'})
 if match:
  assert h['government_role_source_url'] and h['government_role_name_as_captured']==p['candidate_name']
  incumbent_positive.append(p['candidate_name']);mayors.loc[mayors.candidate_key.eq(p['candidate_key']),'incumbency_2026']='現任'
  setfield(p['candidate_key'],'incumbency','現任','內政部名錄與2026可讀官方首長情境交叉支持',h['moi_source_url']+' | '+h['government_role_source_url'],'未將其他人自動編為挑戰者或任期屆滿；代理狀態另列')
  obs(p['candidate_name'],'incumbency','現任',h['government_role_source_url'],'官方首長介紹／近期政府新聞情境，與內政部名錄交叉核對',key=p['candidate_key'])
comparisons=pd.DataFrame(comparisons);assert len(comparisons)==81 and comparisons.candidate_key.is_unique and len(incumbent_positive)==8
for col in ['listed_elected_head_name','matches_listed_elected_head','government_role_name_as_captured','government_role_source_url','acting_head_observed','comparison_status']:
 mayors[col]=mayors.candidate_key.map(comparisons.set_index('candidate_key')[col])
# Third-party claims remain discovery leads. Name mentions on a whole page are not identity/field validation.
plan=json.loads((ROOT/'data/raw/career_expansion_20261019/retrieval_plan.json').read_text())['leads'];wl=[];checked_paths=set();html_cache={}
for x in plan:
 a=x.get('existing_source_capture') or byurl.get(x['source_url'],{})
 path=a.get('path','');status=str(a.get('status',a.get('http_status','')));typ=a.get('content_type','');name_seen='';review='not_downloaded_or_unreadable'
 if path:
  if path not in checked_paths:
   assert (ROOT/path).is_file() and sha(ROOT/path)==a['sha256'],path;checked_paths.add(path)
  if status=='200' and 'html' in typ:
   if path not in html_cache:
    html_cache[path]=norm(BeautifulSoup((ROOT/path).read_bytes().decode('utf-8',errors='replace'),'html.parser').get_text(' ',strip=True))
   name_seen=str(int(norm(x['candidate_name']) in html_cache[path]));review='page_mentions_name_not_identity_or_claim_validation' if name_seen=='1' else 'html_candidate_name_not_found'
  elif status=='200' and (ROOT/path).read_bytes()[:5]==b'%PDF-':review='pdf_bytes_saved_person_field_extraction_pending'
  elif status=='200':review='http200_non_html_non_pdf_body_pending'
 row={k:v for k,v in x.items() if k!='existing_source_capture'};row.update({'http_status':status,'source_file':path,'source_sha256':a.get('sha256',''),'content_type':typ,'page_mentions_candidate_name':name_seen,'review_status':review,'claim_verified':'0','checked_date':'2026-10-19'});wl.append(row)
wl=pd.DataFrame(wl);assert len(wl)==1162 and wl.candidate_key.nunique()==524 and wl.claim_verified.eq('0').all()
print('Mayoral positive head matches:',incumbent_positive,'; third-party leads:',len(wl),'for',wl.candidate_key.nunique(),'people; none treated as validated careers')


Mayoral positive head matches: ['蔣萬安', '張善政', '鍾東錦', '許淑華', '周春米', '王忠銘', '謝國樑', '高虹安'] ; third-party leads: 1162 for 524 people; none treated as validated careers


In [5]:

# Continuation correction using a preserved official page, also reread on the public web.
u='https://www.chcc.gov.tw/member/details.aspx?Parser=99,6,40,,,,142';body=text(u)
assert '陳重嘉' in body and '民國70年1月19日' in body and '彰化縣議會第20屆議員' in body and '彰化縣第17、18、19屆縣議員' in body
key='2026-C2-029';mask=mayors.candidate_key.eq(key);assert mask.sum()==1 and mayors.loc[mask,'candidate_name'].iloc[0]=='陳重嘉'
mayors.loc[mask,['birth_date_verified','birth_date_source_url','birth_date_evidence_status','age_2026_minus_birth_year','first_time_any_elected_office','first_time_any_evidence_status','prior_any_office_evidence_url']]=['1981-01-19',u,'official_council_profile_explicit_full_birth_date_current_role_identity_context','45','0','official_council_profile_lists_previous_elected_council_terms',u]
obs('陳重嘉','birth_date','1981-01-19',u,'官方議會人物頁明載生日；民國70年1月19日轉西元；第20屆／第5選區與同縣市參選身分情境支持',key=key)
obs('陳重嘉','first_time_any_office','0',u,'官方頁明載第17、18、19屆縣議員；非首次任何民選公職；不推定首次縣市長',key=key)
setfield(key,'birth_date','1981-01-19','官方議會人物頁明載完整生日',u,'原頁已保存但先前未接回生日欄，本次讀取正文補入；排除同名其他出生年月者')
setfield(key,'first_time_any_office','0','官方人物頁有過往民選公職經歷支持',u,'曾任第17、18、19屆縣議員；首次縣市長仍未判定')
print('Mayor birthday continuation: 陳重嘉 1981-01-19; complete DOB',int(mayors.birth_date_verified.ne('').sum()),'remaining',int(mayors.birth_date_verified.eq('').sum()))


Mayor birthday continuation: 陳重嘉 1981-01-19; complete DOB 58 remaining 23


In [6]:

# Professional association declaration is tagged separately from government records.
import pypdfium2 as pdfium
u='https://x.miniwork.cc/manager_admin/new_file_download.php?Pact=FileDownLoad&Pval=426898';doc=pdfium.PdfDocument(str(ROOT/byurl[u]['path']));assert len(doc)==1
page=doc[0];tp=page.get_textpage();raw=tp.get_text_range();tp.close();page.close();doc.close();flat=norm(raw);assert '台灣植牙聯盟醫學會' in raw and '110.07.17' in raw
assert norm('8理事温世政男62.05.04世樺牙醫診所院長') in flat
review=json.loads((ROOT/'data/raw/mayor_birth_followup_20261019/web_tool_excerpts.json').read_text());assert '1973年5月4日' in review['sources'][0]['excerpt'] and '世樺牙醫診所院長' in review['sources'][1]['excerpt']
key='2026-C2-033';mask=mayors.candidate_key.eq(key);assert mask.sum()==1 and mayors.loc[mask,'candidate_name'].iloc[0]=='温世政'
mayors.loc[mask,['birth_date_verified','birth_date_source_url','birth_date_evidence_status','age_2026_minus_birth_year']]=['1973-05-04',u,'2021_professional_association_roster_visually_reviewed_birth_and_clinic_identity_corroborated_not_government','53']
obs('温世政','birth_date','1973-05-04',u,'2021-07-17專業學會名冊第1頁第8列視讀確認；出生年月日62.05.04；診所身分及生日與2026媒體人物欄交叉核對；非政府戶籍認證',year='2021',key=key)
setfield(key,'birth_date','1973-05-04','專業學會名冊完整生日已視讀，當屆人物資料交叉支持',u+' | https://www.cna.com.tw/tag/54498/ | https://www.nownews.com/news/6851218','政府與專業學會來源層級分開；媒體交叉材料為web工具摘錄，原HTTP403保留，不假稱成功下載全文')
print('Professional-source birthday continuation: 温世政 1973-05-04; complete DOB',int(mayors.birth_date_verified.ne('').sum()),'remaining',int(mayors.birth_date_verified.eq('').sum()))


Professional-source birthday continuation: 温世政 1973-05-04; complete DOB 59 remaining 22


In [7]:

evidence=pd.DataFrame(evidence)
outputs={
 'data/processed/snapshots/pre_identity_round7_20261019/candidate_master_current.csv':master,
 'data/processed/snapshots/pre_identity_round7_20261019/candidate_fields_current.csv':fields,
 'data/processed/snapshots/pre_identity_round7_20261019/council_background_20261019.csv':council,
 'data/processed/candidate_social_account_evidence_current.csv':accounts,
 'data/processed/meta_candidate_account_worklist_current.csv':work,
 'data/processed/candidate_profiles_round6_20261019.csv':profiles,
 'data/processed/candidate_background_evidence_round6_20261019.csv':evidence,
 'data/processed/mayor_head_comparison_round6_20261019.csv':comparisons,
 'data/processed/candidates_2026_enriched_round6_20261019.csv':mayors,
 'data/processed/council_career_source_worklist_round6_20261019.csv':wl}
checks={}
for p,d in outputs.items():
 target=(ROOT/p).resolve();assert target.is_relative_to((ROOT/'data/processed').resolve()) and p not in PINS
 # Single physical line per CSV record, preserving source line breaks as literal sequences.
 d=d.map(lambda v:str(v).replace('\r','').replace('\n','\\n'))
 d.to_csv(target,index=False,encoding='utf-8-sig');saved=read(p)
 assert list(saved.columns)==list(d.columns) and saved.shape==d.shape
 assert len(target.read_text(encoding='utf-8-sig').splitlines())==len(saved)+1
 key=['candidate_key','field'] if p.endswith('candidate_fields_current.csv') else ['candidate_key','platform'] if p.endswith('meta_candidate_account_worklist_current.csv') else ['candidate_key'] if p in ['data/processed/snapshots/pre_identity_round7_20261019/candidate_master_current.csv','data/processed/snapshots/pre_identity_round7_20261019/council_background_20261019.csv','data/processed/candidate_profiles_round6_20261019.csv','data/processed/mayor_head_comparison_round6_20261019.csv','data/processed/candidates_2026_enriched_round6_20261019.csv'] else []
 if key:assert not saved.duplicated(key).any(),p
 checks[p]={'rows':len(saved),'columns':len(saved.columns),'key':key,'sha256':sha(target)}
summary={'new_profile_people':len(new_profile_keys),'new_profile_names':master[master.candidate_key.isin(new_profile_keys)].candidate_name.tolist(),'remaining_profile_gaps':int((council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum()),'official_additional_birth_date_people':int(council.birth_date_additional_official.ne('').sum()),'new_history_birth_sex_links':linked,'council_history_birth_dates':int(council.birth_date_supported_history.ne('').sum()),'council_history_sex':int(council.sex_supported_history.ne('').sum()),'council_nonfirst_any':int(council.first_time_any_office_supported.eq('0').sum()),'council_nonfirst_council':int(council.first_time_this_office_supported.eq('0').sum()),'new_nonfirst_any':new_nonfirst_any,'social_people':int(accounts.candidate_key.nunique()),'social_candidate_platform_pairs':int(len(accounts[['candidate_key','platform']].drop_duplicates())),'social_by_platform':accounts[['candidate_key','platform']].drop_duplicates().platform.value_counts().to_dict(),'mayor_current_positive':incumbent_positive,'mayor_birth_dates':int(mayors.birth_date_verified.ne('').sum()),'mayor_missing_birth_dates':int(mayors.birth_date_verified.eq('').sum()),'career_lead_rows':len(wl),'career_lead_people':int(wl.candidate_key.nunique()),'outputs':checks}
(ROOT/'data/code/career_round6_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2));print(json.dumps(summary,ensure_ascii=False,indent=2))
# Human-readable current social index, generated from the same saved evidence.
def links(d):return '、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.account_url))) or '待補查'
lines=['# 候選人社群入口（2026-10-19）','','目前涵蓋 **'+str(summary['social_people'])+' 位候選人、'+str(summary['social_candidate_platform_pairs'])+' 組候選人×平台**。'+ '、'.join(f'{k} {v}位' for k,v in summary['social_by_platform'].items())+'。','','本清單為可點選來源入口，不代表已讀取每個目標帳號的內容。未找到不代表沒有帳號；分享連結、共用政黨及媒體頁尾已排除。未蒐集完整貼文、窗口貼文量或追蹤數。','','[本輪逐頁補查筆記](candidate_social_links_followup_20261019.md)的28組入口已併入current CSV；另新增林德建官方人物頁列出的Facebook。','','## 蔣萬安、沈伯洋與薛兆基','']
for name in ['蔣萬安','沈伯洋','薛兆基']:
 p=person(name);d=accounts[accounts.candidate_key.eq(p.candidate_key)]
 lines += [f'### {name}（{p.county_city}／{p.office_type}）','','| Facebook | Instagram | Threads |','| --- | --- | --- |','| '+' | '.join(links(d[d.platform.eq(plat)]) for plat in ['Facebook','Instagram','Threads'])+' |','','來源：'+'、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.source_url))),'']
lines+=['## 其他已找到入口的候選人','','| 候選人 | 縣市／公職 | Facebook | Instagram | Threads | 來源 |','| --- | --- | --- | --- | --- | --- |']
for key,d in accounts.groupby('candidate_key',sort=False):
 p=master[master.candidate_key.eq(key)].iloc[0]
 if p.candidate_name in ['蔣萬安','沈伯洋','薛兆基']:continue
 src='、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.source_url)))
 lines.append('| '+' | '.join([p.candidate_name,p.county_city+'／'+p.office_type]+[links(d[d.platform.eq(plat)]) for plat in ['Facebook','Instagram','Threads']]+[src])+' |')
lines+=['','## 核對界線','','- 薛兆基Instagram／Threads與蔣萬安Threads仍待進一步核對。','- current CSV已整合本輪來源入口，目標帳號內容仍待逐一核對。','- [逐來源帳號證據](../data/processed/candidate_social_account_evidence_current.csv)、[社群工作單](../data/processed/meta_candidate_account_worklist_current.csv)。','- [本輪背景、首長比對與限制](career_round6_20261019.md)。']
(ROOT/'report/candidate_social_links_pre_identity_round7_20261019.md').write_text('\n'.join(lines)+'\n')


{
  "new_profile_people": 16,
  "new_profile_names": [
    "蕭夙玲",
    "劉成謙",
    "張柏詠",
    "何元楷",
    "吳建智咖啡哥",
    "林啟明",
    "陳泓維",
    "徐慈檉",
    "白丞堯",
    "謝秉舟",
    "鄒約翰",
    "潘炩禕",
    "陳傑麟",
    "林德建",
    "鄭素禎",
    "黃舒琳"
  ],
  "remaining_profile_gaps": 733,
  "official_additional_birth_date_people": 4,
  "new_history_birth_sex_links": [
    {
      "candidate_name": "陳泓維",
      "source_file": "data/raw/cec_all_offices/095_elcand.csv",
      "source_row": 256,
      "source_year": 2020,
      "birth_date": "1987-05-08",
      "sex": "男",
      "education_raw": "高中(職)"
    },
    {
      "candidate_name": "林啟明",
      "source_file": "data/raw/cec_all_offices/068_elcand.csv",
      "source_row": 3,
      "source_year": 2014,
      "birth_date": "1962-05-19",
      "sex": "男",
      "education_raw": "專科"
    },
    {
      "candidate_name": "吳建智咖啡哥",
      "source_file": "data/raw/cec_all_offices/116_elcand.csv",
      "source_row": 6,
      "source_year": 2024,
      "birth_

17532